In [1]:
!pip install -q haystack-ai pypdf

from haystack import Pipeline, Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
from haystack.components.converters import PyPDFToDocument

print("Haystack setup completed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.3/734.3 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.0/806.0 kB 40.9 MB/s eta 0:00:00
Haystack setup completed!


In [3]:
!pip install -q reportlab

from reportlab.pdfgen import canvas
import os

os.makedirs("w8_pdfs", exist_ok=True)

pdf_data = {
    "ai.pdf": "Artificial Intelligence is the field of creating systems that can perform tasks requiring human-like intelligence. AI includes machine learning, reasoning, planning, and perception.",

    "ml.pdf": "Machine Learning is a branch of artificial intelligence where computers learn patterns from data. Common approaches include supervised learning, unsupervised learning, and reinforcement learning.",

    "deep_learning.pdf": "Deep Learning uses neural networks with multiple layers to learn complex patterns. It is widely used for image recognition, speech processing, and natural language processing.",

    "nlp.pdf": "Natural Language Processing enables computers to process and understand human language. Applications include translation, sentiment analysis, chatbots, and text classification.",

    "rag.pdf": "Retrieval Augmented Generation combines information retrieval with language generation. A retriever finds relevant documents and a language model uses the retrieved information to generate an answer."
}

for filename, content in pdf_data.items():
    path = os.path.join("w8_pdfs", filename)

    c = canvas.Canvas(path)
    c.setFont("Helvetica", 11)

    y = 750

    for i in range(0, len(content), 90):
        c.drawString(50, y, content[i:i+90])
        y -= 20

    c.save()

print("Created PDFs:")
print(os.listdir("w8_pdfs"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 37.3 MB/s eta 0:00:00
Created PDFs:
['deep_learning.pdf', 'rag.pdf', 'ai.pdf', 'ml.pdf', 'nlp.pdf']


In [4]:
from haystack.components.converters import PyPDFToDocument

converter = PyPDFToDocument()

documents = []

for filename in sorted(os.listdir("w8_pdfs")):
    result = converter.run(
        sources=[os.path.join("w8_pdfs", filename)]
    )
    documents.extend(result["documents"])

print("Number of documents:", len(documents))

document_store = InMemoryDocumentStore()

document_store.write_documents(documents)

bm25_retriever = InMemoryBM25Retriever(
    document_store=document_store,
    top_k=1
)

bm25_pipeline = Pipeline()
bm25_pipeline.add_component("retriever", bm25_retriever)

print("BM25 pipeline created successfully!")
print("Documents indexed:", document_store.count_documents())

Number of documents: 5
BM25 pipeline created successfully!
Documents indexed: 5


In [5]:
questions = [
    "What is Artificial Intelligence?",
    "What is Machine Learning?",
    "What is Deep Learning?",
    "What is Natural Language Processing?",
    "What is Retrieval Augmented Generation?",
    "What are common approaches in Machine Learning?",
    "What are applications of Deep Learning?",
    "What are applications of Natural Language Processing?",
    "How does RAG work?",
    "What is the relationship between AI and Machine Learning?"
]

print("Number of questions:", len(questions))

Number of questions: 10


In [6]:
bm25_results = []

for i, question in enumerate(questions, 1):
    result = bm25_pipeline.run({
        "retriever": {
            "query": question
        }
    })

    docs = result["retriever"]["documents"]

    top_doc = docs[0] if docs else None

    bm25_results.append({
        "question": question,
        "document": top_doc.meta.get("file_path", "Unknown") if top_doc else "No result",
        "content": top_doc.content if top_doc else "",
        "score": top_doc.score if top_doc else 0
    })

    print(f"{i}. {question}")
    print("Retrieved:", bm25_results[-1]["document"])
    print()

1. What is Artificial Intelligence?
Retrieved: ai.pdf

2. What is Machine Learning?
Retrieved: ml.pdf

3. What is Deep Learning?
Retrieved: deep_learning.pdf

4. What is Natural Language Processing?
Retrieved: deep_learning.pdf

5. What is Retrieval Augmented Generation?
Retrieved: rag.pdf

6. What are common approaches in Machine Learning?
Retrieved: ml.pdf

7. What are applications of Deep Learning?
Retrieved: ml.pdf

8. What are applications of Natural Language Processing?
Retrieved: nlp.pdf

9. How does RAG work?
Retrieved: No result

10. What is the relationship between AI and Machine Learning?
Retrieved: ai.pdf



In [7]:
expected_docs = [
    "ai.pdf",
    "ml.pdf",
    "deep_learning.pdf",
    "nlp.pdf",
    "rag.pdf",
    "ml.pdf",
    "deep_learning.pdf",
    "nlp.pdf",
    "rag.pdf",
    "ai.pdf"
]

correct = 0

for i, result in enumerate(bm25_results):
    retrieved = os.path.basename(result["document"])
    expected = expected_docs[i]

    result["relevant"] = int(retrieved == expected)

    if result["relevant"] == 1:
        correct += 1

    print(f"Q{i+1}: {result['relevant']} | Expected: {expected} | Retrieved: {retrieved}")

bm25_precision = correct / len(questions)

print("\nBM25 Precision:", round(bm25_precision, 2))

Q1: 1 | Expected: ai.pdf | Retrieved: ai.pdf
Q2: 1 | Expected: ml.pdf | Retrieved: ml.pdf
Q3: 1 | Expected: deep_learning.pdf | Retrieved: deep_learning.pdf
Q4: 0 | Expected: nlp.pdf | Retrieved: deep_learning.pdf
Q5: 1 | Expected: rag.pdf | Retrieved: rag.pdf
Q6: 1 | Expected: ml.pdf | Retrieved: ml.pdf
Q7: 0 | Expected: deep_learning.pdf | Retrieved: ml.pdf
Q8: 1 | Expected: nlp.pdf | Retrieved: nlp.pdf
Q9: 0 | Expected: rag.pdf | Retrieved: No result
Q10: 1 | Expected: ai.pdf | Retrieved: ai.pdf

BM25 Precision: 0.7


In [9]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

print("Dense retrieval dependencies ready!")

Dense retrieval dependencies ready!


In [10]:
model = SentenceTransformer("all-MiniLM-L6-v2")

dense_documents = []

for doc in documents:
    embedding = model.encode(doc.content).tolist()

    dense_doc = Document(
        content=doc.content,
        meta=doc.meta,
        embedding=embedding
    )

    dense_documents.append(dense_doc)

document_store_dense = InMemoryDocumentStore(
    embedding_similarity_function="cosine"
)

document_store_dense.write_documents(dense_documents)

print("Dense embeddings created successfully!")
print("Documents indexed:", document_store_dense.count_documents())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Dense embeddings created successfully!
Documents indexed: 5


In [11]:
dense_retriever = InMemoryEmbeddingRetriever(
    document_store=document_store_dense,
    top_k=1
)

dense_pipeline = Pipeline()

dense_pipeline.add_component(
    "retriever",
    dense_retriever
)

print("DenseRetriever pipeline created successfully!")

query = "What is Artificial Intelligence?"

query_embedding = model.encode(query).tolist()

result = dense_pipeline.run({
    "retriever": {
        "query_embedding": query_embedding
    }
})

docs = result["retriever"]["documents"]

print("Test question:", query)

if docs:
    print("Retrieved document:", docs[0].meta.get("file_path", "Unknown"))
    print("Similarity score:", docs[0].score)
else:
    print("No document retrieved.")

DenseRetriever pipeline created successfully!
Test question: What is Artificial Intelligence?
Retrieved document: ai.pdf
Similarity score: 0.8450081923880772


In [12]:
dense_results = []

for i, question in enumerate(questions, 1):
    query_embedding = model.encode(question).tolist()

    result = dense_pipeline.run({
        "retriever": {
            "query_embedding": query_embedding
        }
    })

    docs = result["retriever"]["documents"]

    top_doc = docs[0] if docs else None

    dense_results.append({
        "question": question,
        "document": os.path.basename(top_doc.meta.get("file_path", "Unknown")) if top_doc else "No result",
        "score": top_doc.score if top_doc else 0
    })

    print(f"{i}. {question}")
    print("Retrieved:", dense_results[-1]["document"])
    print("Score:", round(dense_results[-1]["score"], 4))
    print()

1. What is Artificial Intelligence?
Retrieved: ai.pdf
Score: 0.845

2. What is Machine Learning?
Retrieved: ml.pdf
Score: 0.7843

3. What is Deep Learning?
Retrieved: deep_learning.pdf
Score: 0.7573

4. What is Natural Language Processing?
Retrieved: nlp.pdf
Score: 0.7917

5. What is Retrieval Augmented Generation?
Retrieved: rag.pdf
Score: 0.7514

6. What are common approaches in Machine Learning?
Retrieved: ml.pdf
Score: 0.7048

7. What are applications of Deep Learning?
Retrieved: deep_learning.pdf
Score: 0.7292

8. What are applications of Natural Language Processing?
Retrieved: nlp.pdf
Score: 0.7412

9. How does RAG work?
Retrieved: rag.pdf
Score: 0.0801

10. What is the relationship between AI and Machine Learning?
Retrieved: ai.pdf
Score: 0.6779



In [13]:
dense_correct = 0

for i, result in enumerate(dense_results):
    retrieved = result["document"]
    expected = expected_docs[i]

    result["relevant"] = int(retrieved == expected)

    if result["relevant"] == 1:
        dense_correct += 1

    print(
        f"Q{i+1}: {result['relevant']} | "
        f"Expected: {expected} | "
        f"Retrieved: {retrieved}"
    )

dense_precision = dense_correct / len(questions)

print("\nDenseRetriever Precision:", round(dense_precision, 2))

Q1: 1 | Expected: ai.pdf | Retrieved: ai.pdf
Q2: 1 | Expected: ml.pdf | Retrieved: ml.pdf
Q3: 1 | Expected: deep_learning.pdf | Retrieved: deep_learning.pdf
Q4: 1 | Expected: nlp.pdf | Retrieved: nlp.pdf
Q5: 1 | Expected: rag.pdf | Retrieved: rag.pdf
Q6: 1 | Expected: ml.pdf | Retrieved: ml.pdf
Q7: 1 | Expected: deep_learning.pdf | Retrieved: deep_learning.pdf
Q8: 1 | Expected: nlp.pdf | Retrieved: nlp.pdf
Q9: 1 | Expected: rag.pdf | Retrieved: rag.pdf
Q10: 1 | Expected: ai.pdf | Retrieved: ai.pdf

DenseRetriever Precision: 1.0


In [14]:
print("=" * 50)
print("RETRIEVAL PRECISION COMPARISON")
print("=" * 50)

print(f"BM25 Precision:          {bm25_precision:.2f}")
print(f"DenseRetriever Precision: {dense_precision:.2f}")

print("\nPercentage:")
print(f"BM25:           {bm25_precision * 100:.0f}%")
print(f"DenseRetriever: {dense_precision * 100:.0f}%")

RETRIEVAL PRECISION COMPARISON
BM25 Precision:          0.70
DenseRetriever Precision: 1.00

Percentage:
BM25:           70%
DenseRetriever: 100%


In [15]:
def retrieve_document(query, top_k=1):
    query_embedding = model.encode(query).tolist()

    result = dense_retriever.run(
        query_embedding=query_embedding,
        top_k=top_k
    )

    docs = result["documents"]

    return [
        {
            "content": doc.content,
            "source": os.path.basename(
                doc.meta.get("file_path", "Unknown")
            ),
            "score": doc.score
        }
        for doc in docs
    ]


# Test the reusable function
test = retrieve_document("What is machine learning?")

print("API retrieval function ready!")
print(test)

API retrieval function ready!
[{'content': 'Machine Learning is a branch of artificial intelligence where computers learn patterns fro\nm data. Common approaches include supervised learning, unsupervised learning, and reinforc\nement learning.\n', 'source': 'ml.pdf', 'score': 0.7843362878512693}]


In [16]:
!pip install -q fastapi uvicorn nest-asyncio

from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Haystack Dense Retrieval API")


class QueryRequest(BaseModel):
    query: str
    top_k: int = 1


@app.get("/")
def home():
    return {
        "message": "Haystack DenseRetriever API is running"
    }


@app.post("/retrieve")
def retrieve(request: QueryRequest):
    results = retrieve_document(
        request.query,
        request.top_k
    )

    return {
        "query": request.query,
        "results": results
    }


print("FastAPI application created successfully!")

FastAPI application created successfully!


In [17]:
test_request = QueryRequest(
    query="What is Natural Language Processing?",
    top_k=1
)

response = retrieve(test_request)

print("API Test")
print("=" * 40)
print("Query:", response["query"])
print("Retrieved:", response["results"][0]["source"])
print("Score:", round(response["results"][0]["score"], 4))

API Test
Query: What is Natural Language Processing?
Retrieved: nlp.pdf
Score: 0.7917


In [18]:
%%writefile app.py
from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer
from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
import os

app = FastAPI(title="Haystack Dense Retrieval API")

model = SentenceTransformer("all-MiniLM-L6-v2")

pdf_data = {
    "ai.pdf": "Artificial Intelligence is the field of creating systems that can perform tasks requiring human-like intelligence. AI includes machine learning, reasoning, planning, and perception.",
    "ml.pdf": "Machine Learning is a branch of artificial intelligence where computers learn patterns from data. Common approaches include supervised learning, unsupervised learning, and reinforcement learning.",
    "deep_learning.pdf": "Deep Learning uses neural networks with multiple layers to learn complex patterns. It is widely used for image recognition, speech processing, and natural language processing.",
    "nlp.pdf": "Natural Language Processing enables computers to process and understand human language. Applications include translation, sentiment analysis, chatbots, and text classification.",
    "rag.pdf": "Retrieval Augmented Generation combines information retrieval with language generation. A retriever finds relevant documents and a language model uses the retrieved information to generate an answer."
}

documents = []

for filename, content in pdf_data.items():
    embedding = model.encode(content).tolist()

    documents.append(
        Document(
            content=content,
            meta={"file_path": filename},
            embedding=embedding
        )
    )

document_store = InMemoryDocumentStore(
    embedding_similarity_function="cosine"
)

document_store.write_documents(documents)

retriever = InMemoryEmbeddingRetriever(
    document_store=document_store,
    top_k=1
)


class QueryRequest(BaseModel):
    query: str
    top_k: int = 1


@app.get("/")
def home():
    return {
        "message": "Haystack DenseRetriever API is running"
    }


@app.post("/retrieve")
def retrieve(request: QueryRequest):
    query_embedding = model.encode(request.query).tolist()

    result = retriever.run(
        query_embedding=query_embedding,
        top_k=request.top_k
    )

    return {
        "query": request.query,
        "results": [
            {
                "source": doc.meta.get("file_path", "Unknown"),
                "content": doc.content,
                "score": doc.score
            }
            for doc in result["documents"]
        ]
    }

Writing app.py


In [19]:
%%writefile requirements.txt
fastapi
uvicorn
haystack-ai
sentence-transformers
torch
pydantic

Writing requirements.txt


In [20]:
%%writefile Procfile
web: uvicorn app:app --host 0.0.0.0 --port $PORT

print("Railway deployment files created!")
print("Files:")
print("- app.py")
print("- requirements.txt")
print("- Procfile")

Writing Procfile


In [21]:
%%writefile Procfile
web: uvicorn app:app --host 0.0.0.0 --port $PORT

Overwriting Procfile


In [22]:
import os

files = ["app.py", "requirements.txt", "Procfile"]

print("Railway deployment files:")
print("=" * 40)

for file in files:
    if os.path.exists(file):
        print(f"✓ {file}")
    else:
        print(f"✗ {file}")

print("\nProcfile content:")
with open("Procfile", "r") as f:
    print(f.read())

Railway deployment files:
✓ app.py
✓ requirements.txt
✓ Procfile

Procfile content:
web: uvicorn app:app --host 0.0.0.0 --port $PORT



In [23]:
import shutil
import os

deployment_files = [
    "app.py",
    "requirements.txt",
    "Procfile"
]

for file in deployment_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"{file} not found")

zip_path = shutil.make_archive(
    "haystack-railway-api",
    "zip",
    root_dir=".",
    base_dir="."
)

print("Deployment package created successfully!")
print(zip_path)

Deployment package created successfully!
/content/haystack-railway-api.zip


In [24]:
import os

print("Current deployment files:")
print("=" * 40)

for file in ["app.py", "requirements.txt", "Procfile"]:
    print(file, "->", "EXISTS" if os.path.exists(file) else "MISSING")

Current deployment files:
app.py -> EXISTS
requirements.txt -> EXISTS
Procfile -> EXISTS


In [25]:
import fastapi
import uvicorn
import haystack
import sentence_transformers

print("FastAPI:", fastapi.__version__)
print("Haystack:", haystack.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("All required packages imported successfully!")

FastAPI: 0.141.1
Haystack: 3.2.0
Sentence Transformers: 5.7.0
All required packages imported successfully!


In [26]:
import subprocess
import sys

server = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "app:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("Uvicorn server started!")
print("Running on http://127.0.0.1:8000")

Uvicorn server started!
Running on http://127.0.0.1:8000


In [27]:
import requests

response = requests.get("http://127.0.0.1:8000/")

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'message': 'Haystack DenseRetriever API is running'}


In [28]:
response = requests.post(
    "http://127.0.0.1:8000/retrieve",
    json={
        "query": "What is Machine Learning?",
        "top_k": 1
    }
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

Status code: 200
Response:
{'query': 'What is Machine Learning?', 'results': [{'source': 'ml.pdf', 'content': 'Machine Learning is a branch of artificial intelligence where computers learn patterns from data. Common approaches include supervised learning, unsupervised learning, and reinforcement learning.', 'score': 0.8036232470111657}]}


In [29]:
import os
import subprocess

if not os.path.exists(".git"):
    subprocess.run(["git", "init"], check=True)

print("Git repository ready!")

Git repository ready!


In [30]:
import subprocess

subprocess.run(
    ["git", "config", "user.name", "YOUR_GITHUB_USERNAME"],
    check=True
)

subprocess.run(
    ["git", "config", "user.email", "YOUR_GITHUB_EMAIL"],
    check=True
)

print("Git configuration completed!")

Git configuration completed!


In [31]:
import subprocess

subprocess.run(
    ["git", "add", "app.py", "requirements.txt", "Procfile"],
    check=True
)

print("Deployment files added to Git!")

Deployment files added to Git!


In [32]:
import subprocess

subprocess.run(
    [
        "git",
        "commit",
        "-m",
        "feat: haystack pipeline — bm25 and dense retrieval"
    ],
    check=True
)

print("Commit created successfully!")

Commit created successfully!


In [33]:
import subprocess

result = subprocess.run(
    ["git", "status"],
    capture_output=True,
    text=True
)

print(result.stdout)

On branch master
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.config/
	__pycache__/
	haystack-railway-api.zip
	sample_data/
	w8_pdfs/

nothing added to commit but untracked files present (use "git add" to track)



In [34]:
import os

required_files = ["app.py", "requirements.txt", "Procfile"]

print("Checking Railway deployment files...")
print("=" * 45)

for file in required_files:
    if os.path.exists(file):
        print(f"✓ {file} exists")
    else:
        print(f"✗ {file} is missing")

Checking Railway deployment files...
✓ app.py exists
✓ requirements.txt exists
✓ Procfile exists


In [36]:

   print("requirements.txt")
print("=" * 40)

with open("requirements.txt", "r") as f:
    print(f.read())

requirements.txt
fastapi
uvicorn
haystack-ai
sentence-transformers
torch
pydantic



In [37]:
print("Procfile")
print("=" * 40)

with open("Procfile", "r") as f:
    print(f.read())

Procfile
web: uvicorn app:app --host 0.0.0.0 --port $PORT



In [38]:
print("app.py information")
print("=" * 40)

with open("app.py", "r") as f:
    app_code = f.read()

print("File exists:", os.path.exists("app.py"))
print("File size:", os.path.getsize("app.py"), "bytes")
print("Contains FastAPI:", "FastAPI" in app_code)
print("Contains DenseRetriever:", "InMemoryEmbeddingRetriever" in app_code)
print("Contains /retrieve endpoint:", '@app.post("/retrieve")' in app_code)

app.py information
File exists: True
File size: 2626 bytes
Contains FastAPI: True
Contains DenseRetriever: True
Contains /retrieve endpoint: True


In [39]:
import requests

response = requests.post(
    "http://127.0.0.1:8000/retrieve",
    json={
        "query": "What is Natural Language Processing?",
        "top_k": 1
    }
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

Status code: 200
Response:
{'query': 'What is Natural Language Processing?', 'results': [{'source': 'nlp.pdf', 'content': 'Natural Language Processing enables computers to process and understand human language. Applications include translation, sentiment analysis, chatbots, and text classification.', 'score': 0.7697435345628516}]}
